# graph-vul: Fusion Ablation — Graph-to-Code Internal Fusion

This notebook runs a fixed 5-experiment ablation suite testing whether injecting
pooled graph embeddings (AST+CFG RGCN) **inside** UniXcoder encoder layers via
gated residual fusion improves vulnerability detection — especially for short
functions (0-64 tokens).

## Experiments

| Tag | Internal Fusion Layers | Late Graph Concat | Description |
|-----|----------------------|-------------------|-------------|
| E0_late_only | None | Yes | Baseline: current late-fusion only |
| E1_last2 | 10, 11 | Yes | Fuse in last 2 layers |
| E2_last4 | 8, 9, 10, 11 | Yes | Fuse in last 4 layers |
| E3_all_layers | 0–11 | Yes | Fuse in all 12 layers |
| E4_last4_no_late_graph | 8, 9, 10, 11 | No | Internal fusion only, no late concat |

All experiments use `USE_GRAPH=True` with identical hyperparameters.
Total runtime: ~5x single run (~6.5 hours). Each experiment saves independently.

## Prerequisites

Same as `colab_finetune.ipynb` — upload to **My Drive/graph-vul/data/**:

```
func_subset.csv, func_splits/assignments.csv,
processed/cwe_embeddings/*.pt, processed/graph_artifacts/*.pt + vocabs
```

**Outputs** (per experiment, written to Drive):
- `data/fusion_ablation/{run_tag}/func_finetune_model.ckpt`
- `data/fusion_ablation/{run_tag}/func_finetune_metrics.json`
- `data/fusion_ablation/{run_tag}/predictions/`
- Final comparison table displayed in the last cell

In [1]:
import subprocess

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU."
    )
print(gpu_info.stdout)

Sun Mar 29 22:28:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/data/graph-vul"
DRIVE_DATA = f"{DRIVE_PROJECT}/data"
DRIVE_HF_CACHE = f"{DRIVE_PROJECT}/hf_cache"

Mounted at /content/drive


In [3]:
!mkdir -p {DRIVE_DATA}
!mkdir -p {DRIVE_HF_CACHE}

In [4]:
import csv
from pathlib import Path

data_root = Path(DRIVE_DATA)
subset_path = data_root / "func_subset.csv"
assignments_path = data_root / "func_splits" / "assignments.csv"
embeddings_dir = data_root / "processed" / "cwe_embeddings"

required_files = [subset_path, assignments_path, embeddings_dir]
missing = [str(path.relative_to(data_root)) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required paths in {DRIVE_DATA}:\n"
        + "\n".join(f"  - {path}" for path in missing)
    )

required_splits = {"train", "val", "test", "unseen_val", "unseen_test"}
with assignments_path.open(newline="") as handle:
    reader = csv.DictReader(handle)
    observed_splits = {row["split"] for row in reader if row.get("split")}

missing_splits = sorted(required_splits - observed_splits)
if missing_splits:
    raise ValueError(
        "assignments.csv is missing required splits:\n"
        + "\n".join(f"  - {split}" for split in missing_splits)
    )

with subset_path.open(newline="") as handle:
    reader = csv.DictReader(handle)
    cwe_ids = sorted({row["cwe_id"].strip() for row in reader if row.get("cwe_id", "").strip()})

missing_embeddings = [cwe_id for cwe_id in cwe_ids if not (embeddings_dir / f"{cwe_id}.pt").exists()]
if missing_embeddings:
    raise FileNotFoundError(
        "Missing CWE embeddings for func_subset.csv entries:\n"
        + "\n".join(f"  - processed/cwe_embeddings/{cwe_id}.pt" for cwe_id in missing_embeddings)
    )

print(f"Validated dataset root: {data_root}")
print(f"Found {len(cwe_ids)} CWE embedding files referenced by func_subset.csv.")
print(f"Observed splits: {sorted(observed_splits)}")

Validated dataset root: /content/drive/MyDrive/data/graph-vul/data
Found 89 CWE embedding files referenced by func_subset.csv.
Observed splits: ['test', 'train', 'unseen_test', 'unseen_val', 'val']


In [5]:
# Validate graph artifacts (always required for fusion ablation)
graph_dir = data_root / "processed" / "graph_artifacts"
node_vocab = graph_dir / "node_type_vocab.json"
edge_vocab = graph_dir / "edge_type_vocab.json"

missing_graph = []
if not graph_dir.exists():
    missing_graph.append(str(graph_dir.relative_to(data_root)))
if not node_vocab.exists():
    missing_graph.append(str(node_vocab.relative_to(data_root)))
if not edge_vocab.exists():
    missing_graph.append(str(edge_vocab.relative_to(data_root)))
if missing_graph:
    raise FileNotFoundError(
        f"Graph artifacts required for fusion ablation but missing in {DRIVE_DATA}:\n"
        + "\n".join(f"  - {p}" for p in missing_graph)
    )

graph_pts = list(graph_dir.glob("*.pt"))
print(f"Graph artifacts validated: {len(graph_pts)} .pt files found in {graph_dir}")

Graph artifacts validated: 1733 .pt files found in /content/drive/MyDrive/data/graph-vul/data/processed/graph_artifacts


In [6]:
import os

os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_HF_CACHE

In [7]:
!git clone https://github.com/noamdwc/graph-vul.git /content/graph-vul
%pip install -q pytorch-lightning transformers sentencepiece pandas numpy \
    scikit-learn torchmetrics wandb torch-geometric

Cloning into '/content/graph-vul'...
remote: Enumerating objects: 796, done.
remote: Counting objects: 100% (796/796), done.
remote: Compressing objects: 100% (444/444), done.
remote: Total 796 (delta 333), reused 677 (delta 216), pack-reused 0 (from 0)
Receiving objects: 100% (796/796), 2.35 MiB | 7.73 MiB/s, done.
Resolving deltas: 100% (333/333), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.6 MB/s eta 0:00:00


In [8]:
import os
import shutil
from pathlib import Path

repo_data = "/content/graph-vul/data"

if os.path.islink(repo_data):
    os.unlink(repo_data)
elif os.path.isdir(repo_data):
    shutil.rmtree(repo_data)

os.symlink(DRIVE_DATA, repo_data)
assert Path(repo_data, "func_subset.csv").exists(), "Symlink verification failed"
print(f"Symlinked {repo_data} -> {DRIVE_DATA}")

Symlinked /content/graph-vul/data -> /content/drive/MyDrive/data/graph-vul/data


## Weights & Biases

The training script logs to WandB. Run the cell below to authenticate.
To skip WandB logging, uncomment the `WANDB_MODE` line instead.

In [9]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: the-doctor (the-doctor-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Ablation Configuration

Shared hyperparameters (identical across all experiments) and the 5-experiment
ablation suite. Each experiment varies only `fusion_layers` and `final_graph_concat`.

**OOM?** Reduce `BATCH_SIZE` to 4 and increase `GRAD_ACCUM` to 16 to keep effective batch size at 64.

In [10]:
# Shared hyperparameters
NUM_EPOCHS = 10
BATCH_SIZE = 64
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
GRAD_ACCUM = 8
SEED = 42

# Ablation suite — each dict is one experiment
EXPERIMENTS = [
    {
        "run_tag": "E0_late_only",
        "fusion_layers": None,
        "final_graph_concat": True,
    },
    {
        "run_tag": "E1_last2",
        "fusion_layers": [10, 11],
        "final_graph_concat": True,
    },
    {
        "run_tag": "E2_last4",
        "fusion_layers": [8, 9, 10, 11],
        "final_graph_concat": True,
    },
    {
        "run_tag": "E3_all_layers",
        "fusion_layers": list(range(12)),
        "final_graph_concat": True,
    },
    {
        "run_tag": "E4_last4_no_late_graph",
        "fusion_layers": [8, 9, 10, 11],
        "final_graph_concat": False,
    },
]

print(f"Configured {len(EXPERIMENTS)} experiments:")
for exp in EXPERIMENTS:
    fl = exp["fusion_layers"]
    fl_str = f"{len(fl)} layers" if fl else "none"
    print(f"  {exp['run_tag']}: fusion={fl_str}, late_concat={exp['final_graph_concat']}")

Configured 5 experiments:
  E0_late_only: fusion=none, late_concat=True
  E1_last2: fusion=2 layers, late_concat=True
  E2_last4: fusion=4 layers, late_concat=True
  E3_all_layers: fusion=12 layers, late_concat=True
  E4_last4_no_late_graph: fusion=4 layers, late_concat=False


## Run Ablation Suite

The cell below loops over all experiments, training each model and extracting
length-bucketed metrics via the existing prediction export + post-analysis pipeline.
Each experiment writes to its own subdirectory under `fusion_ablation/`.

In [ ]:
import json
import logging
import math
import sys
import time

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
sys.path.insert(0, "/content/graph-vul")

from pathlib import Path

from graph_vul.train_finetune import train
from graph_vul.export_finetune_predictions import export_split_predictions
from graph_vul.model.func_finetune_lit_module import FuncFinetuneLitModule
from graph_vul.analysis.post_experiment_analysis import (
    canonicalize_prediction_frame,
    stratified_metrics,
    DEFAULT_LENGTH_BUCKETS,
)
from transformers import AutoTokenizer
from graph_vul.config import UNIXCODER_MODEL_NAME

ABLATION_ROOT = Path(DRIVE_DATA) / "fusion_ablation"
summary_rows = []

for i, exp in enumerate(EXPERIMENTS):
    run_tag = exp["run_tag"]
    print(f"\n{'='*60}")
    print(f"Experiment {i+1}/{len(EXPERIMENTS)}: {run_tag}")
    print(f"{'='*60}")

    exp_dir = ABLATION_ROOT / run_tag
    exp_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = exp_dir / "func_finetune_model.ckpt"
    metrics_path = exp_dir / "func_finetune_metrics.json"
    pred_dir = exp_dir / "predictions"

    # Train
    t0 = time.time()
    train(
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        encoder_lr=ENCODER_LR,
        head_lr=HEAD_LR,
        grad_accum=GRAD_ACCUM,
        seed=SEED,
        use_graph=True,
        fusion_layers=exp["fusion_layers"],
        final_graph_concat=exp["final_graph_concat"],
        checkpoint_path=ckpt_path,
        metrics_path=metrics_path,
        run_tag=run_tag,
    )
    train_minutes = (time.time() - t0) / 60.0
    print(f"Training completed in {train_minutes:.1f} minutes")

    # Load metrics
    with open(metrics_path) as f:
        metrics = json.load(f)

    # Export predictions for test and unseen_test
    tokenizer = AutoTokenizer.from_pretrained(UNIXCODER_MODEL_NAME)
    model = FuncFinetuneLitModule.load_from_checkpoint(
        str(ckpt_path), map_location="cuda" if __import__("torch").cuda.is_available() else "cpu"
    )
    model.eval()

    for split in ("test", "unseen_test"):
        export_split_predictions(
            model=model,
            tokenizer=tokenizer,
            split=split,
            output_dir=str(pred_dir),
            batch_size=BATCH_SIZE,
        )

    # Extract length-bucket metrics from test predictions
    row = {
        "run_tag": run_tag,
        "test_f1": metrics.get("test_f1"),
        "unseen_test_f1": metrics.get("unseen_test_f1"),
        "train_minutes": round(train_minutes, 1),
    }

    test_pred_path = pred_dir / "test_predictions.csv"
    if test_pred_path.exists():
        test_df = pd.read_csv(test_pred_path)
        test_df, _ = canonicalize_prediction_frame(test_df, split_name="test")
        token_buckets = DEFAULT_LENGTH_BUCKETS["token_length"]
        strat = stratified_metrics(test_df, "token_length", token_buckets)

        if not strat.empty:
            for _, brow in strat.iterrows():
                bucket = str(brow.name).replace("[", "").replace(")", "").replace(", ", "_").replace("inf", "inf")
                # Normalize bucket names: "[0, 64)" -> "0_64"
                parts = str(brow.name).strip("[]()").split(",")
                if len(parts) == 2:
                    lo = parts[0].strip().replace(".0", "")
                    hi = parts[1].strip().replace(".0", "").replace("inf", "inf")
                    bucket_key = f"{lo}_{hi}"
                else:
                    bucket_key = bucket

                row[f"{bucket_key}_f1"] = brow.get("f1")
                if bucket_key == "0_64":
                    row["0_64_precision"] = brow.get("precision")
                    row["0_64_recall"] = brow.get("recall")

    summary_rows.append(row)
    print(f"Summary row: {row}")

print(f"\nAll {len(summary_rows)} experiments completed.")

INFO:lightning_fabric.utilities.seed:Seed set to 42



Experiment 1/5: E0_late_only


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/unixcoder-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_

...

Trainable params: 126 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 126 M                                                                                                
Total estimated model params size (MB): 505                                                                        
Modules in train mode: 31                                                                                          
Modules in eval mode: 228                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 228 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

## Results Summary

Compact comparison table across all experiments. Key columns:
- **test_f1 / unseen_test_f1**: overall and generalization performance
- **0_64_***: short-function metrics (primary question)
- **64_128, 128_256, 256_512**: longer function buckets for regression check

In [ ]:
import pandas as pd

display_cols = [
    "run_tag",
    "test_f1",
    "unseen_test_f1",
    "0_64_f1",
    "0_64_precision",
    "0_64_recall",
    "64_128_f1",
    "128_256_f1",
    "256_512_f1",
    "train_minutes",
]

results_df = pd.DataFrame(summary_rows)

# Reorder columns, keeping only those that exist
cols = [c for c in display_cols if c in results_df.columns]
results_df = results_df[cols].set_index("run_tag")

# Format numeric columns to 4 decimal places (except train_minutes)
fmt = {c: "{:.4f}" for c in results_df.columns if c != "train_minutes"}
if "train_minutes" in results_df.columns:
    fmt["train_minutes"] = "{:.1f}"

print("Fusion Ablation Results")
print("=" * 80)
display(results_df.style.format(fmt).highlight_max(subset=["test_f1", "unseen_test_f1", "0_64_f1"], color="lightgreen"))